# Text Preprocessing Pipeline with spaCy and NLTK

**Objective**: Build a comprehensive text preprocessing pipeline using both spaCy and NLTK, comparing their approaches for Italian and English text processing.

**Duration**: ~35 minutes

**Learning Outcomes**:
- Understand classical NLP preprocessing steps
- Compare spaCy vs NLTK approaches
- Handle multilingual text processing
- Explore POS tagging, lemmatization, and NER

---

## 1. Setup and Installation

First, let's ensure all required libraries are installed and load the language models.

In [ ]:
# Installation commands (run in terminal if needed)
# pip install spacy nltk matplotlib seaborn pandas
# python -m spacy download en_core_web_sm
# python -m spacy download it_core_news_sm

import spacy
import nltk
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tag import pos_tag

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')

print("✓ Libraries imported successfully")

In [ ]:
# Load spaCy language models
try:
    nlp_en = spacy.load("en_core_web_sm")
    nlp_it = spacy.load("it_core_news_sm")
    print("✓ spaCy models loaded successfully")
    print(f"English model: {nlp_en.meta['name']} v{nlp_en.meta['version']}")
    print(f"Italian model: {nlp_it.meta['name']} v{nlp_it.meta['version']}")
except IOError as e:
    print(f"❌ Error loading spaCy models: {e}")
    print("Please run: python -m spacy download en_core_web_sm it_core_news_sm")

## 2. Sample Texts for Analysis

We'll use parallel texts in English and Italian to demonstrate multilingual processing capabilities.

In [ ]:
# Sample texts for preprocessing
english_text = """
Natural Language Processing (NLP) has revolutionized how we interact with technology. 
Companies like OpenAI, Google, and Meta have developed sophisticated language models 
that can understand and generate human-like text. These systems process millions of 
words daily, handling tasks from translation to content generation. However, they 
still struggle with nuanced understanding, context-dependent meanings, and factual accuracy.
The field continues evolving rapidly, with new architectures and techniques emerging regularly.
"""

italian_text = """
Il Natural Language Processing (NLP) ha rivoluzionato il modo in cui interagiamo con la tecnologia.
Aziende come OpenAI, Google e Meta hanno sviluppato modelli linguistici sofisticati che possono 
comprendere e generare testo simile a quello umano. Questi sistemi processano milioni di parole 
quotidianamente, gestendo compiti dalla traduzione alla generazione di contenuti. Tuttavia, 
faticano ancora con la comprensione sfumata, i significati dipendenti dal contesto e l'accuratezza 
fattuale. Il campo continua ad evolversi rapidamente, con nuove architetture e tecniche che emergono 
regolarmente.
"""

print("Sample texts loaded:")
print(f"English text: {len(english_text)} characters")
print(f"Italian text: {len(italian_text)} characters")

## 3. Basic Text Cleaning and Normalization

Let's start with fundamental text cleaning operations that are often the first step in any preprocessing pipeline.

In [ ]:
def clean_text(text):
    """Basic text cleaning function"""
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove leading/trailing whitespace
    text = text.strip()
    # Convert to lowercase (optional - depends on use case)
    text_lower = text.lower()
    
    return text, text_lower

# Clean both texts
en_clean, en_lower = clean_text(english_text)
it_clean, it_lower = clean_text(italian_text)

print("Original English (first 100 chars):")
print(repr(english_text[:100]))
print("\nCleaned English (first 100 chars):")
print(repr(en_clean[:100]))

## 4. Tokenization Comparison: NLTK vs spaCy

Let's compare how NLTK and spaCy handle tokenization for both languages.

In [ ]:
def compare_tokenization(text, language='english'):
    """Compare NLTK and spaCy tokenization"""
    
    # NLTK tokenization
    nltk_tokens = word_tokenize(text, language=language)
    nltk_sentences = sent_tokenize(text, language=language)
    
    # spaCy tokenization
    nlp = nlp_en if language == 'english' else nlp_it
    doc = nlp(text)
    spacy_tokens = [token.text for token in doc]
    spacy_sentences = [sent.text.strip() for sent in doc.sents]
    
    return {
        'nltk_tokens': nltk_tokens,
        'nltk_sentences': nltk_sentences,
        'spacy_tokens': spacy_tokens,
        'spacy_sentences': spacy_sentences
    }

# Compare tokenization for English
en_tokens = compare_tokenization(en_clean, 'english')

print("=== ENGLISH TOKENIZATION COMPARISON ===")
print(f"NLTK word tokens: {len(en_tokens['nltk_tokens'])}")
print(f"spaCy word tokens: {len(en_tokens['spacy_tokens'])}")
print(f"NLTK sentences: {len(en_tokens['nltk_sentences'])}")
print(f"spaCy sentences: {len(en_tokens['spacy_sentences'])}")

print("\nFirst 10 NLTK tokens:")
print(en_tokens['nltk_tokens'][:10])
print("\nFirst 10 spaCy tokens:")
print(en_tokens['spacy_tokens'][:10])

In [ ]:
# Compare tokenization for Italian
it_tokens = compare_tokenization(it_clean, 'italian')

print("=== ITALIAN TOKENIZATION COMPARISON ===")
print(f"NLTK word tokens: {len(it_tokens['nltk_tokens'])}")
print(f"spaCy word tokens: {len(it_tokens['spacy_tokens'])}")
print(f"NLTK sentences: {len(it_tokens['nltk_sentences'])}")
print(f"spaCy sentences: {len(it_tokens['spacy_sentences'])}")

print("\nFirst 10 NLTK tokens:")
print(it_tokens['nltk_tokens'][:10])
print("\nFirst 10 spaCy tokens:")
print(it_tokens['spacy_tokens'][:10])

## 5. Stop Word Removal

Compare stop word handling between NLTK and spaCy for both languages.

In [ ]:
def analyze_stopwords(tokens, language='english'):
    """Analyze stop words using both NLTK and spaCy"""
    
    # NLTK stop words
    lang_code = 'english' if language == 'english' else 'italian'
    nltk_stopwords = set(stopwords.words(lang_code))
    
    # spaCy stop words
    nlp = nlp_en if language == 'english' else nlp_it
    spacy_stopwords = nlp.Defaults.stop_words
    
    # Filter stop words
    nltk_filtered = [token for token in tokens if token.lower() not in nltk_stopwords]
    spacy_filtered = [token for token in tokens if token.lower() not in spacy_stopwords]
    
    # Find stop words in text
    found_nltk_stops = [token for token in tokens if token.lower() in nltk_stopwords]
    found_spacy_stops = [token for token in tokens if token.lower() in spacy_stopwords]
    
    return {
        'original_tokens': len(tokens),
        'nltk_filtered': len(nltk_filtered),
        'spacy_filtered': len(spacy_filtered),
        'nltk_stopwords_found': found_nltk_stops,
        'spacy_stopwords_found': found_spacy_stops,
        'nltk_stopwords_total': len(nltk_stopwords),
        'spacy_stopwords_total': len(spacy_stopwords)
    }

# Analyze English stop words
en_stopword_analysis = analyze_stopwords(en_tokens['nltk_tokens'], 'english')

print("=== ENGLISH STOP WORDS ANALYSIS ===")
print(f"Original tokens: {en_stopword_analysis['original_tokens']}")
print(f"After NLTK stop word removal: {en_stopword_analysis['nltk_filtered']}")
print(f"After spaCy stop word removal: {en_stopword_analysis['spacy_filtered']}")
print(f"NLTK stop words vocabulary size: {en_stopword_analysis['nltk_stopwords_total']}")
print(f"spaCy stop words vocabulary size: {en_stopword_analysis['spacy_stopwords_total']}")
print(f"\nNLTK stop words found: {en_stopword_analysis['nltk_stopwords_found'][:10]}")
print(f"spaCy stop words found: {en_stopword_analysis['spacy_stopwords_found'][:10]}")

In [ ]:
# Analyze Italian stop words
it_stopword_analysis = analyze_stopwords(it_tokens['nltk_tokens'], 'italian')

print("=== ITALIAN STOP WORDS ANALYSIS ===")
print(f"Original tokens: {it_stopword_analysis['original_tokens']}")
print(f"After NLTK stop word removal: {it_stopword_analysis['nltk_filtered']}")
print(f"After spaCy stop word removal: {it_stopword_analysis['spacy_filtered']}")
print(f"NLTK stop words vocabulary size: {it_stopword_analysis['nltk_stopwords_total']}")
print(f"spaCy stop words vocabulary size: {it_stopword_analysis['spacy_stopwords_total']}")
print(f"\nNLTK stop words found: {it_stopword_analysis['nltk_stopwords_found'][:10]}")
print(f"spaCy stop words found: {it_stopword_analysis['spacy_stopwords_found'][:10]}")

## 6. Stemming vs Lemmatization

Compare stemming and lemmatization approaches, highlighting their differences and use cases.

In [ ]:
def compare_stemming_lemmatization(tokens, language='english'):
    """Compare stemming and lemmatization results"""
    
    # NLTK stemming (Porter Stemmer for English)
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    
    # NLTK lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token.lower()) for token in tokens]
    
    # spaCy lemmatization
    nlp = nlp_en if language == 'english' else nlp_it
    doc = nlp(' '.join(tokens))
    spacy_lemmatized = [token.lemma_ for token in doc]
    
    return {
        'original': tokens,
        'stemmed': stemmed_tokens,
        'nltk_lemmatized': lemmatized_tokens,
        'spacy_lemmatized': spacy_lemmatized
    }

# Test with specific examples
test_words = ['running', 'runs', 'ran', 'better', 'worse', 'mice', 'children', 'feet']
comparison = compare_stemming_lemmatization(test_words)

print("=== STEMMING vs LEMMATIZATION COMPARISON ===")
df = pd.DataFrame({
    'Original': comparison['original'],
    'Stemmed': comparison['stemmed'],
    'NLTK Lemma': comparison['nltk_lemmatized'],
    'spaCy Lemma': comparison['spacy_lemmatized']
})

print(df.to_string(index=False))

## 7. Part-of-Speech (POS) Tagging

Explore POS tagging capabilities of both libraries.

In [ ]:
def analyze_pos_tags(text, language='english'):
    """Compare POS tagging between NLTK and spaCy"""
    
    # NLTK POS tagging
    tokens = word_tokenize(text)
    nltk_pos = pos_tag(tokens)
    
    # spaCy POS tagging
    nlp = nlp_en if language == 'english' else nlp_it
    doc = nlp(text)
    spacy_pos = [(token.text, token.pos_, token.tag_) for token in doc]
    
    return {
        'nltk_pos': nltk_pos,
        'spacy_pos': spacy_pos
    }

# Analyze POS tags for a sample sentence
sample_sentence = "The sophisticated language models developed by OpenAI are revolutionizing natural language processing."
pos_analysis = analyze_pos_tags(sample_sentence)

print("=== POS TAGGING COMPARISON ===")
print("Sample sentence:", sample_sentence)
print("\nNLTK POS Tags:")
for word, tag in pos_analysis['nltk_pos']:
    print(f"{word:<15} {tag}")

print("\nspaCy POS Tags:")
for word, pos, tag in pos_analysis['spacy_pos']:
    print(f"{word:<15} {pos:<6} {tag}")

## 8. Named Entity Recognition (NER)

Explore NER capabilities, which are more advanced in spaCy than NLTK.

In [ ]:
def analyze_named_entities(text, language='english'):
    """Extract named entities using spaCy"""
    nlp = nlp_en if language == 'english' else nlp_it
    doc = nlp(text)
    
    entities = []
    for ent in doc.ents:
        entities.append({
            'text': ent.text,
            'label': ent.label_,
            'description': spacy.explain(ent.label_),
            'start': ent.start_char,
            'end': ent.end_char
        })
    
    return entities

# Analyze entities in English text
en_entities = analyze_named_entities(en_clean)

print("=== ENGLISH NAMED ENTITIES ===")
for entity in en_entities:
    print(f"{entity['text']:<20} {entity['label']:<10} {entity['description']}")

print("\n=== ITALIAN NAMED ENTITIES ===")
it_entities = analyze_named_entities(it_clean, 'italian')
for entity in it_entities:
    print(f"{entity['text']:<20} {entity['label']:<10} {entity['description'] or 'N/A'}")

## 9. Complete Preprocessing Pipeline

Now let's build a complete preprocessing pipeline that combines all the techniques we've explored.

In [ ]:
class TextPreprocessor:
    """Complete text preprocessing pipeline"""
    
    def __init__(self, language='english'):
        self.language = language
        self.nlp = nlp_en if language == 'english' else nlp_it
        self.stemmer = PorterStemmer()
        self.lemmatizer = WordNetLemmatizer()
        
        # Load stop words
        lang_code = 'english' if language == 'english' else 'italian'
        self.stop_words = set(stopwords.words(lang_code))
    
    def clean_text(self, text):
        """Basic text cleaning"""
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        # Remove leading/trailing whitespace
        text = text.strip()
        return text
    
    def preprocess(self, text, steps=None):
        """Complete preprocessing pipeline"""
        
        if steps is None:
            steps = ['clean', 'tokenize', 'lowercase', 'remove_stopwords', 'lemmatize']
        
        result = {'original': text}
        processed_text = text
        
        # Step 1: Clean text
        if 'clean' in steps:
            processed_text = self.clean_text(processed_text)
            result['cleaned'] = processed_text
        
        # Step 2: Tokenize
        if 'tokenize' in steps:
            doc = self.nlp(processed_text)
            tokens = [token.text for token in doc if not token.is_punct and not token.is_space]
            result['tokens'] = tokens
        
        # Step 3: Lowercase
        if 'lowercase' in steps and 'tokens' in result:
            tokens = [token.lower() for token in tokens]
            result['lowercase'] = tokens
        
        # Step 4: Remove stop words
        if 'remove_stopwords' in steps and 'lowercase' in result:
            tokens = [token for token in tokens if token not in self.stop_words]
            result['no_stopwords'] = tokens
        
        # Step 5: Lemmatize
        if 'lemmatize' in steps:
            doc = self.nlp(' '.join(tokens) if 'no_stopwords' in result else processed_text)
            lemmatized = [token.lemma_ for token in doc if not token.is_punct and not token.is_space]
            result['lemmatized'] = lemmatized
        
        # Step 6: Stem (alternative to lemmatization)
        if 'stem' in steps:
            tokens_to_stem = result.get('no_stopwords', result.get('lowercase', result.get('tokens', [])))
            stemmed = [self.stemmer.stem(token) for token in tokens_to_stem]
            result['stemmed'] = stemmed
        
        return result

# Test the preprocessing pipeline
preprocessor_en = TextPreprocessor('english')
preprocessor_it = TextPreprocessor('italian')

# Process English text
en_processed = preprocessor_en.preprocess(en_clean)

print("=== ENGLISH PREPROCESSING PIPELINE ===")
for step, result in en_processed.items():
    if step == 'original':
        continue
    print(f"\n{step.upper()}:")
    if isinstance(result, list):
        print(f"Length: {len(result)}")
        print(f"Sample: {result[:10]}")
    else:
        print(f"Length: {len(result)}")
        print(f"Sample: {result[:100]}...")

In [ ]:
# Process Italian text
it_processed = preprocessor_it.preprocess(it_clean)

print("=== ITALIAN PREPROCESSING PIPELINE ===")
for step, result in it_processed.items():
    if step == 'original':
        continue
    print(f"\n{step.upper()}:")
    if isinstance(result, list):
        print(f"Length: {len(result)}")
        print(f"Sample: {result[:10]}")
    else:
        print(f"Length: {len(result)}")
        print(f"Sample: {result[:100]}...")

## 10. Visualization and Analysis

Let's visualize the preprocessing results to better understand the impact of each step.

In [ ]:
# Create visualization of preprocessing pipeline impact
def visualize_preprocessing_impact(processed_results, language):
    """Visualize the impact of each preprocessing step"""
    
    steps = ['original', 'tokens', 'lowercase', 'no_stopwords', 'lemmatized']
    lengths = []
    labels = []
    
    for step in steps:
        if step in processed_results:
            if step == 'original':
                length = len(processed_results[step].split())
                labels.append('Original Words')
            else:
                length = len(processed_results[step])
                labels.append(step.replace('_', ' ').title())
            lengths.append(length)
    
    # Create bar plot
    plt.figure(figsize=(12, 6))
    bars = plt.bar(labels, lengths, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
    plt.title(f'Preprocessing Pipeline Impact - {language.title()}', fontsize=14, fontweight='bold')
    plt.xlabel('Preprocessing Step', fontsize=12)
    plt.ylabel('Number of Tokens', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    
    # Add value labels on bars
    for bar, length in zip(bars, lengths):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                str(length), ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return lengths, labels

# Visualize English preprocessing
en_lengths, en_labels = visualize_preprocessing_impact(en_processed, 'English')

In [ ]:
# Visualize Italian preprocessing
it_lengths, it_labels = visualize_preprocessing_impact(it_processed, 'Italian')

# Create comparison chart
plt.figure(figsize=(14, 8))
x = range(len(en_labels))
width = 0.35

plt.bar([i - width/2 for i in x], en_lengths, width, label='English', alpha=0.8, color='#1f77b4')
plt.bar([i + width/2 for i in x], it_lengths, width, label='Italian', alpha=0.8, color='#ff7f0e')

plt.title('Preprocessing Pipeline Comparison: English vs Italian', fontsize=16, fontweight='bold')
plt.xlabel('Preprocessing Step', fontsize=12)
plt.ylabel('Number of Tokens', fontsize=12)
plt.xticks(x, en_labels, rotation=45, ha='right')
plt.legend(fontsize=12)

# Add value labels
for i, (en_len, it_len) in enumerate(zip(en_lengths, it_lengths)):
    plt.text(i - width/2, en_len + 1, str(en_len), ha='center', va='bottom', fontsize=10)
    plt.text(i + width/2, it_len + 1, str(it_len), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 11. Performance Comparison: NLTK vs spaCy

Let's measure the performance differences between NLTK and spaCy for various preprocessing tasks.

In [ ]:
import time

def benchmark_preprocessing(text, iterations=100):
    """Benchmark preprocessing performance"""
    
    results = {}
    
    # NLTK tokenization benchmark
    start_time = time.time()
    for _ in range(iterations):
        tokens = word_tokenize(text)
    results['nltk_tokenization'] = (time.time() - start_time) / iterations
    
    # spaCy tokenization benchmark
    start_time = time.time()
    for _ in range(iterations):
        doc = nlp_en(text)
        tokens = [token.text for token in doc]
    results['spacy_tokenization'] = (time.time() - start_time) / iterations
    
    # NLTK POS tagging benchmark
    tokens = word_tokenize(text)
    start_time = time.time()
    for _ in range(iterations):
        pos_tags = pos_tag(tokens)
    results['nltk_pos_tagging'] = (time.time() - start_time) / iterations
    
    # spaCy full pipeline benchmark
    start_time = time.time()
    for _ in range(iterations):
        doc = nlp_en(text)
        # Access POS tags to ensure processing
        pos_tags = [(token.text, token.pos_) for token in doc]
    results['spacy_full_pipeline'] = (time.time() - start_time) / iterations
    
    return results

# Benchmark with sample text
benchmark_text = en_clean
performance_results = benchmark_preprocessing(benchmark_text, 50)

print("=== PERFORMANCE BENCHMARK RESULTS (50 iterations) ===")
print("Average time per operation:")
for operation, time_taken in performance_results.items():
    print(f"{operation:<25} {time_taken*1000:.2f} ms")

# Visualize performance comparison
plt.figure(figsize=(10, 6))
operations = list(performance_results.keys())
times = [t*1000 for t in performance_results.values()]  # Convert to milliseconds

bars = plt.bar(operations, times, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.title('Performance Comparison: NLTK vs spaCy', fontsize=14, fontweight='bold')
plt.xlabel('Operation', fontsize=12)
plt.ylabel('Average Time (ms)', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add value labels on bars
for bar, time_val in zip(bars, times):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            f'{time_val:.2f}ms', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 12. Key Takeaways and Best Practices

Let's summarize what we've learned about classical text preprocessing.

### Summary of Findings

**Tokenization**:
- spaCy generally produces more tokens due to better handling of contractions and punctuation
- NLTK is faster for basic tokenization but less linguistically sophisticated
- Both handle multilingual text, but spaCy has better language-specific models

**Stop Word Removal**:
- Different libraries have different stop word lists
- Impact varies significantly by text type and language
- Consider domain-specific stop words for specialized applications

**Lemmatization vs Stemming**:
- Lemmatization is more accurate but slower
- spaCy's lemmatization is generally superior to NLTK's
- Stemming is sufficient for some applications (e.g., search)

**Performance**:
- NLTK is faster for individual operations
- spaCy is more efficient when multiple operations are needed
- spaCy provides more linguistic information in a single pass

### When Classical Preprocessing Still Matters

1. **Information Retrieval Systems**: Search engines, document indexing
2. **Traditional Machine Learning**: Feature-based models (SVM, Naive Bayes)
3. **Resource-Constrained Environments**: Where computational efficiency is critical
4. **Domain-Specific Applications**: Legal, medical, financial text processing
5. **Data Cleaning**: Preparing text for human annotation or analysis

### When It Doesn't Matter

1. **Modern Language Models**: Transformers handle raw text effectively
2. **End-to-End Systems**: Neural models learn optimal representations
3. **Multilingual Applications**: Subword tokenization handles diverse languages
4. **Context-Dependent Tasks**: Where word order and context are crucial

## 13. Exercise: Build Your Own Pipeline

Now it's your turn! Create a preprocessing pipeline for a specific use case.

In [ ]:
# Exercise: Create a preprocessing pipeline for social media text
social_media_text = """
@OpenAI just released GPT-4! 🤖 It's amazing how far #AI has come. 
Can't wait to try it out! 😍 https://openai.com/gpt-4 
RT if you're excited too! #MachineLearning #NLP #Tech
"""

# TODO: Build a preprocessing pipeline that:
# 1. Handles mentions (@OpenAI) and hashtags (#AI)
# 2. Removes or processes URLs
# 3. Handles emojis appropriately
# 4. Deals with social media-specific language

def preprocess_social_media(text):
    """
    Your task: Implement a social media text preprocessing function
    
    Consider:
    - Should you keep or remove mentions?
    - How to handle hashtags?
    - What to do with URLs?
    - How to process emojis?
    """
    # Your code here
    pass

# Test your function
# processed_social = preprocess_social_media(social_media_text)
# print("Original:", social_media_text)
# print("Processed:", processed_social)

## Next Steps

In the next notebook, we'll explore modern tokenization techniques used by Large Language Models, including:

- Subword tokenization algorithms (BPE, WordPiece, SentencePiece)
- Comparison of different tokenizer implementations
- Impact of tokenization on model performance and costs
- Practical considerations for multilingual applications

**Resources for Further Reading**:
- [spaCy Documentation](https://spacy.io/)
- [NLTK Documentation](https://www.nltk.org/)
- [Jurafsky & Martin, Chapter 2](https://web.stanford.edu/~jurafsky/slp3/)
- [Text Preprocessing in Python](https://towardsdatascience.com/text-preprocessing-in-python-steps-tools-and-examples-bf025f872908)